In [1]:
import spacy
from spacy.tokens import DocBin
from collections import Counter

nlp = spacy.blank("en")

doc_bin = DocBin().from_disk("data/ner/train.spacy")

labels = Counter()

for doc in doc_bin.get_docs(nlp.vocab):
    for ent in doc.ents:
        labels[ent.label_] += 1

print(labels)


/home/abhi/miniforge3/envs/nlp_intern/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Counter({'PARTY': 1834, 'CONTRACT_DATE': 524, 'RESTRICTION': 108, 'LICENSE_TERM': 70, 'TERMINATION': 58, 'PAYMENT_TERM': 57, 'LIABILITY': 35, 'IP_OWNERSHIP': 29, 'LEGAL': 27})


In [2]:
len(labels)

9

In [3]:
# Add /ner_model to the end of your path
nlp = spacy.load("/home/abhi/Projects/Fintech-Intelligent-Document-Processing-NLP/artifacts/ner_web_gpu")


In [4]:
ner = nlp.get_pipe("ner")
print(sorted(ner.labels))
print(len(ner.labels))

['CONTRACT_DATE', 'IP_OWNERSHIP', 'LEGAL', 'LIABILITY', 'LICENSE_TERM', 'PARTY', 'PAYMENT_TERM', 'RESTRICTION', 'TERMINATION']
9


In [5]:
print(nlp.pipe_names)

['tok2vec', 'ner', 'sentencizer']


In [6]:
text = """
This Agreement is made on the 7th day of September, 1999 
between Electric City Corp. and ABC Distribution Ltd.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

7th day of September, 1999 -> CONTRACT_DATE
Electric City Corp. -> PARTY
ABC Distribution Ltd. -> PARTY


In [7]:
text = """
Distributor agrees to purchase no fewer than 100,000 units annually at a total minimum commitment of $5,000,000.
"""

doc = nlp(text)


for ent in doc.ents:
    if ent.label_ != "PARTIES":
        print(ent.text, "->", ent.label_)


In [8]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

payment_patterns = [

    [{"LOWER": "shall"}, {"LOWER": "pay"}],
    [{"LOWER": "agrees"}, {"LOWER": "to"}, {"LOWER": "pay"}],
    [{"LOWER": "must"}, {"LOWER": "pay"}],
    [{"LOWER": "minimum"}, {"LOWER": "commitment"}],
    [{"LOWER": "license"}, {"LOWER": "fee"}],
    [{"LOWER": "annual"}, {"LOWER": "fee"}],
    [{"LOWER": "royalty"}],
]

matcher.add("PAYMENT_RULE", payment_patterns)


In [9]:
def rule_payment_extractor(doc):

    matches = matcher(doc)
    spans = []

    for _, start, end in matches:

        sentence = doc[start:end].sent
        spans.append(sentence.text)

    return list(set(spans))   # deduplicate


In [10]:
def extract_payment_hybrid(doc):

    ner_spans = [
        ent.text for ent in doc.ents
        if ent.label_ == "PAYMENT_TERM"
    ]

    rule_spans = rule_payment_extractor(doc)

    return list(set(ner_spans + rule_spans))


In [11]:
text="""This Agreement is entered into as of March 3, 2021, by and between AlphaNet Solutions, Inc. and BetaLogix Corporation. Licensee shall pay Licensor a minimum annual royalty of $1,250,000 during each Contract Year of the Term. In addition, Distributor agrees to purchase no fewer than 100,000 units annually at a total minimum commitment of $5,000,000. Any disputes shall be resolved under the laws of New York.
"""

In [12]:
doc = nlp(text)

print("NER PAYMENT_TERM:")
for ent in doc.ents:
    if ent.label_ == "PAYMENT_TERM":
        print(" -", ent.text)

print("\nRULE PAYMENT:")
for span in rule_payment_extractor(doc):
    print(" -", span)

print("\nHYBRID OUTPUT:")
for span in extract_payment_hybrid(doc):
    print(" -", span)


NER PAYMENT_TERM:

RULE PAYMENT:
 - In addition, Distributor agrees to purchase no fewer than 100,000 units annually at a total minimum commitment of $5,000,000.
 - Licensee shall pay Licensor a minimum annual royalty of $1,250,000 during each Contract Year of the Term.

HYBRID OUTPUT:
 - In addition, Distributor agrees to purchase no fewer than 100,000 units annually at a total minimum commitment of $5,000,000.
 - Licensee shall pay Licensor a minimum annual royalty of $1,250,000 during each Contract Year of the Term.
